# A synthetic CT with MAISI-v2

No input scan: the model makes a CT volume from settings.

In [ ]:
# Install the package into this notebook's kernel once; remove the # to run it.
# %pip install nalvera

A key typed into a cell is saved in the notebook file and in IPython's history file; `nalvera.login()` with no arguments asks for it instead.

In [ ]:
import nalvera

nalvera.login("you@university.org", "nvk_...")

## The settings the task accepts

In [ ]:
ct = next(task for model in nalvera.models() if model["id"] == "nv_generate_ctmr"
          for task in model["tasks"] if task["id"] == "rflow-ct")
ct["generation"]

## Queue the generation

This spends credits from the budget "personal"; free tasks are used first. With a label, running the cell again returns the same task and queues nothing.

In [ ]:
task = nalvera.generate(
    model="nv_generate_ctmr",           # required. See: nalvera.models()
    task="rflow-ct",                    # required. Options: "rflow-ct" | "rflow-mr-brain"
    params={
        "mode": "image-only",           # required. Options: "image-only" | "image-mask"
        "output_size": [256, 256, 128],  # required. Options: [X, X, Z], X from 128 | 256 |
                                         # 384 | 512, Z from 128 | 256
        "spacing": [1.5, 1.5, 2.0],     # required. Options: [S, S, Sz] in mm, each 0.5 to 5.0;
                                        # X times S at least 256 mm
        "seed": 7,                      # default: 0. Options: 0 to 2147483647
    },
    budget="personal",                  # default: None (free tasks first, then personal, or
                                        # asks). Options: "personal" | "lab:<id>" | "free" | "ask"
    label="phantom-01",                 # default: None (the file name starts with the task id)
)
task

## Wait and save it

In [ ]:
finished = nalvera.wait(task)
finished

In [ ]:
paths = nalvera.download(finished, out="generated")  # default: "results"
paths

## Open it

In [ ]:
import nibabel as nib

volume = nib.load(paths[0])
volume.shape, volume.header.get_zooms()